# GraphRAG 질문·답변 테스트

질문을 입력하고 우리 불새 지식그래프와 LangGraph가 찾은 답변·출처를 확인하는 노트북입니다.

**사용 순서:** 위에서부터 셀을 한 번 실행한 뒤, 질문 셀의 내용을 바꾸고 질문 셀과 실행 셀을 다시 실행하세요. VS Code에서 프로젝트의 `.venv` 파이썬 커널을 선택해 주세요.

- Neo4j는 읽기 전용으로 조회하며 팀원의 `B_` 데이터는 제외합니다.
- 기본 질문은 **2025년 연간 개별 재무 비교**라서 OpenAI API를 호출하지 않습니다.
- 뉴스 질문과 대표·행사 질문의 LLM 질문 분석에는 OpenAI API 비용이 발생합니다. 원문 재조회가 필요하면 답변 생성과 검토를 최대 한 번 반복합니다.


## 1. 실행 환경 준비

프로젝트 위치와 `.env`를 찾고 필요한 함수를 불러옵니다. 아래 출력은 설정 여부만 표시하며 비밀번호와 API 키 값은 출력하지 않습니다.


In [ ]:
import os
import sys
from pathlib import Path

# 노트북을 프로젝트 폴더 또는 data_code 폴더에서 열었을 때 프로젝트 루트를 찾습니다.
current = Path.cwd().resolve()
candidates = [current, *current.parents, current / "bulsae"]
project_root = next(
    (path for path in candidates
     if (path / "pyproject.toml").is_file() and (path / "src" / "bulsae").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("불새 프로젝트 폴더를 찾지 못했어요. VS Code에서 bulsae 폴더를 열고 실행해 주세요.")

sys.path.insert(0, str(project_root / "src"))

from dotenv import load_dotenv
from openai import OpenAI
from bulsae.graphrag.pipeline import driver
from bulsae.graphrag.document_search import fetch_evidence, fetch_strategy_evidence, fetch_document_context, generate_claims, review_claims
from bulsae.graphrag.entity_resolution import resolve_question
from bulsae.graphrag.financial_comparison import fetch_financials
from bulsae.graphrag.company_information import fetch_company_business
from bulsae.graphrag.question_analysis import analyze_question
from bulsae.graphrag.multihop_search import fetch_executive_events
from bulsae.graphrag.graphs import QueryDeps, build_query_graph

load_dotenv(project_root / ".env")
print("파이썬:", sys.executable)
for key in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE", "OPENAI_API_KEY"):
    print(key + ":", "설정됨" if os.getenv(key) else "설정되지 않음")
print("환경 준비 완료!")


## 2. 질문 처리 함수 준비

이 함수는 기존 프로젝트 코드를 연결합니다. 질문에서 기업·인물을 찾고, 관련 근거를 검색한 다음, 답변 또는 보류 결과를 돌려줍니다. **이 셀은 함수를 준비하기만 하며 DB나 LLM을 호출하지 않습니다.**


In [ ]:
def ask_question(question, limit=200, model="gpt-4o-mini"):
    """질문 하나를 LangGraph에 전달하고 전체 결과를 반환합니다."""
    if not question.strip():
        raise ValueError("질문을 입력해 주세요.")

    # Neo4j 연결과 OpenAI 클라이언트는 State에 넣지 않고 함수로 전달합니다.
    with driver() as db:
        clients = []

        def get_client():
            # 재무 비교만 할 때는 OpenAI 클라이언트를 만들지 않습니다.
            if not clients:
                clients.append(OpenAI(timeout=90, max_retries=1))
            return clients[0]

        deps = QueryDeps(
            identify=lambda text: resolve_question(db, text),
            search=lambda name, kind, size: fetch_evidence(db, name, kind, size),
            financial=lambda names: fetch_financials(db, names),
            strategy=lambda name, year: fetch_strategy_evidence(db, name, year),
            context=lambda ids: fetch_document_context(db, ids),
            company_info=lambda name: fetch_company_business(db, name),
            analyze=lambda text, name: analyze_question(get_client(), model, text, name),
            multihop=lambda name, role, year: fetch_executive_events(db, name, role, year),
            generate=lambda text, evidence: generate_claims(get_client(), model, text, evidence),
            review=lambda text, claims, evidence: review_claims(get_client(), model, text, claims, evidence),
        )
        graph = build_query_graph(deps)
        return graph.invoke({"question": question, "limit": limit})

print("질문 처리 함수 준비 완료!")


## 3. 질문 입력

아래 `question` 안의 문장을 원하는 질문으로 바꾸고 이 셀을 실행하세요. **이 셀은 질문만 저장하고 아직 검색하지 않습니다.**


In [ ]:
question = "아모레퍼시픽과 이니스프리의 2025년 매출과 영업이익을 비교해줘"

# 아래처럼 질문을 바꿀 수 있어요. 뉴스 질문은 LLM API 비용이 발생합니다.
# question = "김승환 아모레퍼시픽 대표가 참석한 행사는 뭐야?"
# question = "아모레퍼시픽의 대표이사가 2026년에 참석한 행사는 무엇인가요?"
# question = "아모레퍼시픽과 이니스프리가 참여한 행사를 각각 비교해줘"

print("질문:", question)


## 4. 질문 실행 및 답변 확인

아래 셀을 실행하면 실제 Neo4j를 조회합니다. 답변에 있는 출처와 처리 상태를 함께 확인하세요. `insufficient_evidence`는 비교할 근거가 부족해서 답변을 보류했다는 뜻입니다.


In [ ]:
result = ask_question(question)

if result.get("entities"):
    print("비교 대상:", ", ".join(item["name"] for item in result["entities"]))
elif result.get("name"):
    print("검색 대상:", result["name"], "(인물)" if result.get("kind") == "person" else "(기업)")

print("\n=== 답변 ===")
print(result.get("answer", "답변을 찾지 못했어요."))
print("\n처리 상태:", result.get("status"))


## 현재 가능한 범위

- **재무 조회와 비교:** 한 기업 또는 기업 2~4곳의 2025년 연간 개별 매출액·영업이익. 다른 연도, 분기, 전년 대비, 이익률은 아직 비교하지 않습니다.
- **뉴스 질문:** 회사별 원문 근거를 모아서 보여 줍니다. 근거가 부족하거나 대상을 구분할 수 없으면 보류합니다.
- **자동 추출 관계:** 기사에서 찾은 후보이며 사람의 최종 검토가 끝난 사실은 아닙니다.
- 이름이 없거나 인물 여러 명을 비교하는 질문은 현재 대상을 다시 물어봅니다.
